# Treino SFT — tool-use calibrado + reasoning (QLoRA + Unsloth)

**Branch `phronesis-thinking`.** GPU alvo: **L4** (Colab Pro).

Ordem obrigatória:
1. Rodar com `USE_GUINEA_PIG = True` (Qwen3-1.7B) para validar o pipeline (loss desce, salva, carrega).
2. Só então treinar o alvo deste experimento, `MODEL_SIZE = "8b"`.
   **Antes, estimar créditos e reportar ao usuário; > 15 créditos → parar e discutir.**

Antes de rodar: a célula abaixo dá `git clone`/`git pull` do repo `Conatus-Phronesis` usando o secret
`GH_TOKEN` já configurado no Colab (ícone de chave 🔑 na barra lateral) — não precisa de upload manual nem Drive.
Este notebook também serve para a **Fase 0** (baseline): pule para a última seção sem treinar.

In [ ]:
# Puxa (ou atualiza) o repositório com data/clean/train.jsonl, configs/ e src/,
# garante o branch phronesis-thinking (dataset com <think> — o main não tem isso),
# e autentica no Hugging Face (evita rate limit anônimo / permite modelos privados).
# GH_TOKEN e HF_TOKEN vêm do Colab Secrets (ícone de chave na sidebar).
import os
from google.colab import userdata
from huggingface_hub import login

GH_TOKEN = userdata.get("GH_TOKEN")
GH_USER = "devlucascfarias"
REPO_NAME = "Conatus-Phronesis"
BRANCH = "phronesis-thinking"
REPO = f"/content/{REPO_NAME}"
REPO_URL = f"https://{GH_TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git"

if os.path.isdir(f"{REPO}/.git"):
    !cd {REPO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO}

!cd {REPO} && git branch --show-current

del GH_TOKEN, REPO_URL  # não deixa o token solto na sessão do notebook além do necessário

login(token=userdata.get("HF_TOKEN"))

In [ ]:
%pip install -q unsloth
%pip install -q --no-deps trl peft accelerate bitsandbytes
import torch
print(torch.cuda.get_device_name(0))

In [ ]:
# ---- Configuração (espelha configs/train_config.yaml) ----
USE_GUINEA_PIG = True   # PRIMEIRO True (1.7B, barato); depois False para o alvo 8B
MODEL_SIZE = "8b"

_MODEL_BY_SIZE = {
    "4b": "Qwen/Qwen3-4B-Instruct-2507",  # referência sem thinking
    "8b": "Qwen/Qwen3-8B",                # alvo híbrido deste branch
}

# REPO já foi definido na célula de clone/pull acima (/content/Conatus-Phronesis)
MODEL = "Qwen/Qwen3-1.7B" if USE_GUINEA_PIG else _MODEL_BY_SIZE[MODEL_SIZE]
TAG = "1p7b" if USE_GUINEA_PIG else MODEL_SIZE
MAX_SEQ_LEN = 4096
TRAIN_FILE = f"{REPO}/data/clean/train.jsonl"

In [ ]:
# Renderiza data/clean/dataset.jsonl no chat template do Qwen3 + máscara de loss (seção 4.3),
# gerando o train.jsonl que a célula seguinte consome. Os blocos <think> dos itens difíceis
# ficam dentro do span treinável do assistant; turnos sem reasoning continuam normais.
!cd {REPO} && python src/build_dataset.py data/clean/dataset.jsonl --out data/clean/train.jsonl --model {MODEL} --max-len {MAX_SEQ_LEN}

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
from datasets import load_dataset

# train.jsonl vem do build_dataset.py e já contém o campo 'text' renderizado
# via tokenizer.apply_chat_template (nunca concatenar strings na mão).
dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")
dataset = dataset.remove_columns([c for c in dataset.column_names if c != "text"])
print(dataset, "\n", dataset[0]["text"][:600])

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # batch efetivo ~16
        num_train_epochs=1,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        logging_steps=5,
        seed=42,
        output_dir=f"outputs_{TAG}",
        report_to="none",
    ),
)

# Alinhado à máscara da seção 4.3: loss só nos turnos do assistant;
# system/user/<tool_response> (renderizado dentro de um turno user pelo template) ficam fora.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Verificação da máscara antes de treinar (equivalente ao --show-masks do build_dataset.py)
sample = trainer.train_dataset[0]
ids, labels = sample["input_ids"], sample["labels"]
trainable = [t for t, l in zip(ids, labels) if l != -100]
print("TREINÁVEL:", tokenizer.decode(trainable)[:500])
assert any(l != -100 for l in labels), "máscara zerou tudo — conferir template"

In [ ]:
stats = trainer.train()
print(stats)

In [ ]:
# Salvar adapter + merged 16-bit (para a Fase 4)
model.save_pretrained(f"{REPO}/outputs/adapter_{TAG}")
tokenizer.save_pretrained(f"{REPO}/outputs/adapter_{TAG}")
model.save_pretrained_merged(f"{REPO}/outputs/merged_{TAG}", tokenizer, save_method="merged_16bit")
print("Salvo. Teste de sanidade de geração abaixo.")

In [ ]:
# Sanidade: checkpoint carrega e gera no template correto
import json
FastLanguageModel.for_inference(model)
tools = json.load(open(f"{REPO}/configs/tools.json", encoding="utf-8"))["tools"]
msgs = [{"role": "user", "content": "Quanto tá o dólar hoje?"}]
prompt = tokenizer.apply_chat_template(msgs, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
# repetition_penalty: greedy puro entra em loop de repeticao variando um numero a cada linha
# (evade no_repeat_ngram_size, que so bloqueia n-gramas identicos) — visto no eval do held-out.
out = model.generate(**inputs, max_new_tokens=2048, do_sample=False,
                     repetition_penalty=1.15, no_repeat_ngram_size=8,
                     pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## Avaliação (Fase 0 baseline / Fase 4 treinado)

Fase 0 (sem treino): rode a célula abaixo direto, sem executar as células de treino.
Fase 4: aponte `--adapter` para o adapter salvo.

Ao final da sessão, **anote os créditos consumidos** e atualize a tabela da seção 6 do PLANO.md.

In [ ]:
# Fase 0 — baseline do modelo cru:
!cd {REPO} && python src/eval_harness.py --model {MODEL} --out outputs/baseline_metrics.json

# Fase 4 — modelo treinado (adapter descomentado por padrao; ja causou confusao uma vez rodar sem):
!cd {REPO} && python src/eval_harness.py --model {MODEL} --adapter outputs/adapter_{TAG} --out outputs/trained_metrics.json

## Fase 5 — demo do agente com busca REAL na internet

Roda o **8B treinado** (em memória, na mesma sessão) num loop de agente de verdade: gera → detecta `<tool_call>` → **executa** → devolve o resultado → gera de novo.

- `web_search` usa o **Ollama web search** (secret `OLLAMA_SEARCH_KEY` do Colab); cai para DuckDuckGo se falhar.
- `python_sandbox` executa o código de verdade (subprocess isolado, timeout 5s, imports na whitelist).

Testa os 3 fluxos: pergunta volátil (busca), cálculo (sandbox) e conversa (sem tool).

In [ ]:
%pip install -q ddgs   # fallback de busca; provedor principal e o Ollama (OLLAMA_SEARCH_KEY)

import os, sys, json
from google.colab import userdata
sys.path.insert(0, f"{REPO}/src")

# chave de busca do Ollama (secret do Colab) -> ambiente, pra o executor enxergar
os.environ["OLLAMA_SEARCH_KEY"] = userdata.get("OLLAMA_SEARCH_KEY")

from inference_loop import run_agent
tools = json.load(open(f"{REPO}/configs/tools.json", encoding="utf-8"))["tools"]

FastLanguageModel.for_inference(model)   # usa o modelo treinado que ja esta em memoria

def generate(messages):
    prompt = tokenizer.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # repetition_penalty: greedy puro entra em loop de repeticao variando um numero a cada linha
    # (evade no_repeat_ngram_size sozinho) — visto no eval do held-out (autovalores, EDO, etc.)
    out = model.generate(**inputs, max_new_tokens=2048, do_sample=False, no_repeat_ngram_size=8,
                         repetition_penalty=1.15,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for pergunta in [
    "Quanto ta o dolar hoje?",          # volatil -> web_search real
    "Quanto e 37,5% de 18.420?",        # calculo -> python_sandbox
    "Me conta uma curiosidade aleatoria.",  # conversa -> sem tool
]:
    print("="*70)
    print(f"voce> {pergunta}\n")
    run_agent([{"role": "user", "content": pergunta}], generate, verbose=True)

In [ ]:
# Bateria de avaliação manual — reusa generate()/tools/run_agent já definidos na célula acima.
# Cobre: os 3 casos que falharam ao vivo antes do batch_022 (conferir se o corretivo pegou)
# + uma amostra de outras camadas pra visão geral de qualidade.

perguntas_avaliacao = [
    # --- regressões documentadas, checando se batch_022 corrigiu ---
    "Quanto é 37,5% de 18.420?",                         # camada 2: resposta final tinha alucinado 18.422/18.423
    "Quanto tá o dólar hoje?",                            # camada 1: tinha inventado "Banco Central" além da fonte real
    "Me conta uma curiosidade sobre polígonos regulares.", # camada C: ângulo de 1000 lados saiu errado (179,999... em vez de 179,64)
    "Calcule 62,5% de 4.960.",                            # camada 2: sem contexto de dinheiro — checa se ainda inventa "reais"
    "Quem venceu o jogo do Brasil ontem?",                # camada 1: checa se cita só fontes reais quando há mais de uma

    # --- amostra geral de outras camadas ---
    "Qual é o segundo maior planeta do sistema solar?",   # camada 0: deve responder seco, sem tool
    "Quem era o presidente do Brasil na Copa de 94?",     # camada 0.5: parece atual, mas é histórico
    "Discussão séria: pizza com abacaxi é crime ou aceitável?",  # camada C: conversa, sem tool
    "Resolva \\(2x^2-11x+12=0\\) por Bhaskara.",          # camada 2: matemática de rotina, deve chamar sandbox
]

for pergunta in perguntas_avaliacao:
    print("=" * 70)
    print(f"voce> {pergunta}\n")
    run_agent([{"role": "user", "content": pergunta}], generate, verbose=True)